<a href="https://colab.research.google.com/github/fazmina11/fazmina-codeboosters-2026/blob/main/Day/day7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install chromadb sentence-transformers -q
import warnings
warnings.filterwarnings('ignore')

print("ChromaDB installed Successfully!!")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 66.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 20.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 66.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 72.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 15.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 4.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the s

In [ ]:
import pandas as pd
import numpy as np
#New Lib : sentence-transforms
#SentenceTransfomers : the class we use to load the embedding model
from sentence_transformers import SentenceTransformer

#New Lib : chromaDB
#ChromaDB : the vector database package
import chromadb

print("All libraries imported sucessfully")
print(f"ChromaDB version :{chromadb.__version__}")


All libraries imported sucessfully
ChromaDB version :1.5.9


In [ ]:
#KEY WORD SEARCH

documents = [
    "ETL is data transformation pipeline",
    "A vehicle is a mode of transportation",
    "Cars and trucks are popular automoblies",
    "Extracts, Transform and Load",
    "MAchine learning trains models o data"
]

query_keyword = "vehicle"

print("="*60)
print(f"KEYWORD SEARCH for: {query_keyword}")
print("="*60)

for i,doc in enumerate(documents):
    if query_keyword.lower() in doc.lower():
      print((f" FOUND [doc_{i}]: {doc}"))
    else:
      print(f" MISSED [doc_{i}]: {doc}")

print()
print("PROBLEM:doc_2 talks about 'Cars and trucks' - which ARE vehicles!")
print("but keyword seach MISSED it because it searched for the exact word 'vehicle'.")



KEYWORD SEARCH for: vehicle
 MISSED [doc_0]: ETL is data transformation pipeline
 FOUND [doc_1]: A vehicle is a mode of transportation
 MISSED [doc_2]: Cars and trucks are popular automoblies
 MISSED [doc_3]: Extracts, Transform and Load
 MISSED [doc_4]: MAchine learning trains models o data

PROBLEM:doc_2 talks about 'Cars and trucks' - which ARE vehicles!
but keyword seach MISSED it because it searched for the exact word 'vehicle'.


In [ ]:
#Note : First run doenloads the model (~8-MB). This is one-time download
print("Loading embedding model...(may take 2-3 mins on first run)")
model = SentenceTransformer('all-MiniLM-L6-v2')

#The model is now loaded and ready to convert text to embeddigns
print("Model loaded successfully !!")
print(f"Model produces vector of size {model.get_sentence_embedding_dimension()} dimentions")

Loading embedding model...(may take 2-3 mins on first run)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded successfully !!
Model produces vector of size 384 dimentions


In [ ]:
sentence = "ELT is used to clean and transform data"

embedding = model.encode(sentence)


print(f"input sentence: '{sentence}'")
print()
print(f"Embedding type :{type(embedding)}")

print(f'Embedding shape :{embedding.shape}')
print(f"First 10 numbers : {embedding[:10].round(4)}")

print(f"Min value : {embedding.min():.4f}")
print(f"Max value : {embedding.max():.4f}")

input sentence: 'ELT is used to clean and transform data'

Embedding type :<class 'numpy.ndarray'>
Embedding shape :(384,)
First 10 numbers : [-0.0699  0.034   0.0146 -0.0433  0.114  -0.0833  0.0151  0.0441  0.0228
  0.0013]
Min value : -0.1707
Max value : 0.1450


In [ ]:
#Semantic search

# Using cosine similarity in util module
from sentence_transformers import util

print(f"Query: {query_keyword}\n")

for i,doc in enumerate(documents):
  print(f"Doc{i}: {doc}")
  embedding_query = model.encode(query_keyword.lower())
  embedding_doc = model.encode(doc.lower())
  similarity = util.cos_sim(embedding_query, embedding_doc).item()
  if similarity > 0.4:
    print(f" FOUND [doc_{i}]: {doc} (similarity={similarity:.2f})\n")
  else:
    print(f" MISSED [doc_{i}]: {doc} (similarity={similarity:.2f})\n")



Query: vehicle

Doc0: ETL is data transformation pipeline
 MISSED [doc_0]: ETL is data transformation pipeline (similarity=0.04)

Doc1: A vehicle is a mode of transportation
 FOUND [doc_1]: A vehicle is a mode of transportation (similarity=0.73)

Doc2: Cars and trucks are popular automoblies
 FOUND [doc_2]: Cars and trucks are popular automoblies (similarity=0.52)

Doc3: Extracts, Transform and Load
 MISSED [doc_3]: Extracts, Transform and Load (similarity=0.09)

Doc4: MAchine learning trains models o data
 MISSED [doc_4]: MAchine learning trains models o data (similarity=0.19)



In [ ]:
sentences = [
    "ETL is used to clean and transform data",
    "Data Transformation is a key pipeline step",
    "The sky is blue and the clouds are white",
]

embeddings = model.encode(sentences)

print(f"Number of sentences:{len(sentences)}")
print(f"Shape of embedding array: {embeddings.shape}")

print()
print("Each row is one sentence's embedding: ")
for i,sent in enumerate(sentences):
  print(f" Sentence{i}:shape ={embeddings[i].shape}, first 5 values{embeddings[i][:5]}")

Number of sentences:3
Shape of embedding array: (3, 384)

Each row is one sentence's embedding: 
 Sentence0:shape =(384,), first 5 values[-0.07843897  0.0540922   0.02240578 -0.0389119   0.02206615]
 Sentence1:shape =(384,), first 5 values[-0.04657807  0.05854008 -0.00124817 -0.03335287 -0.04608623]
 Sentence2:shape =(384,), first 5 values[0.04593757 0.07167661 0.08381855 0.05318833 0.04961346]


In [ ]:
sentence =[
    "Machine Learning is used to predict the ouput based on trained data",
    "Types : Supervised and Unsupervised Learning",
    "I enjoy learning in the classroom",
]

embedding = model.encode(sentence)
query_keyword = "Machine Learning"
print(f"Query: {query_keyword}\n")

for i,doc in enumerate(sentence):
  print(f"Doc{i}: {doc}")
  embedding_query = model.encode(query_keyword.lower())
  embedding_doc = model.encode(doc.lower())
  similarity = util.cos_sim(embedding_query, embedding_doc).item()
  if similarity > 0.4:
    print(f" FOUND [doc_{i}]: {doc} (similarity={similarity:.2f})\n")
  else:
    print(f" MISSED [doc_{i}]: {doc} (similarity={similarity:.2f})\n")



Query: Machine Learning

Doc0: Machine Learning is used to predict the ouput based on trained data
 FOUND [doc_0]: Machine Learning is used to predict the ouput based on trained data (similarity=0.64)

Doc1: Types : Supervised and Unsupervised Learning
 FOUND [doc_1]: Types : Supervised and Unsupervised Learning (similarity=0.48)

Doc2: I enjoy learning in the classroom
 MISSED [doc_2]: I enjoy learning in the classroom (similarity=0.12)



In [ ]:
chroma_client =chromadb.Client()
collection=chroma_client.get_or_create_collection("demo_notes")
print("ChromaDB client created (in-memory mode)")


print(f"Collection name:demo_notes ")
print(f"Documents in collection :{collection.count()}")

ChromaDB client created (in-memory mode)
Collection name:demo_notes 
Documents in collection :0


In [ ]:
sample_docs = [
    "ETL is data transformation pipeline",
    "A vehicle is a mode of transportation",
    "Cars and trucks are popular automoblies",
    "Extracts, Transform and Load",
]

sample_ids = ['doc001', 'doc002', 'doc003', 'doc004']

sample_metadata = [
    {"subject": "Data Engineering", "topic": "ETL"},
    {"subject": "Transportation", "topic": "Vehicles"},
    {"subject": "Transportation", "topic": "Automobiles"},
    {"subject": "Data Engineering", "topic": "Data Transformation"},

]

sample_embeddings = model.encode(sample_docs).tolist()

collection.add(
    documents=sample_docs,
    embeddings=sample_embeddings, # Pass the generated embeddings
    ids=sample_ids,
    metadatas=sample_metadata
)

print("Documents added successfully")
print(f"Number of documents in collection: {collection.count()}")

Documents added successfully
Number of documents in collection: 5


In [ ]:
query="How do I clean and prepare data?"

results=collection.query(
    query_texts=[query],
    n_results=3
)

print("RESULT KEYS AVAILABLE:")
print(list(results.keys()))

/root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:00<00:00, 107MiB/s]


RESULT KEYS AVAILABLE:
['ids', 'embeddings', 'documents', 'uris', 'included', 'data', 'metadatas', 'distances']


In [ ]:
print(f"Query :'{query}'")
print("="*60)
print()

matched_docs = results['documents'][0]
matched_ids = results['ids'][0]
matched_distance = results['distances'][0]
matched_metadata = results['metadatas'][0]

for rank,(doc , doc_id,dist,meta)in enumerate(zip(matched_docs,matched_ids,matched_distance,matched_metadata)):
  print(f"Rank:{rank}|ID:{doc_id}|Distance:{dist:.2f}")
  print(f"Subject:{meta['subject']}|Topic:{meta['topic']}")
  print(f"Document:{doc}\n")

Query :'How do I clean and prepare data?'

Rank:0|ID:doc001|Distance:1.62
Subject:Data Engineering|Topic:ETL
Document:ETL is data transformation pipeline

Rank:1|ID:doc005|Distance:1.70
Subject:Machine Learning|Topic:Models
Document:Machine learning trains models o data

Rank:2|ID:doc004|Distance:1.71
Subject:Data Engineering|Topic:Data Transformation
Document:Extracts, Transform and Load



In [ ]:
filtered_results = collection.query(
    query_texts=[query],
    n_results=3,
    where={"topic": "ETL"}
)



print(f"FILTERED QUERY: {query}")
print("Filter only Data Engineering documents")
print("="*60)

for rank, (doc, dist, meta) in enumerate(zip(
    filtered_results['documents'][0],
    filtered_results['distances'][0],
    filtered_results['metadatas'][0]),start=1):
  print(f"Rank: {rank} | Distance: {dist:.2f} | Subject: {meta['subject']}")
  print(f"{doc}\n")

FILTERED QUERY: How do I clean and prepare data?
Filter only Data Engineering documents
Rank: 1 | Distance: 1.62 | Subject: Data Engineering
ETL is data transformation pipeline



In [ ]:
print("DISTANCE TO SIMILARITY CONVERSION")
print(f"{'Distance':<15}{'Similarity':<15}{'Interpretation':<20}")
distances=[0.05,0.20,0.40,0.65,0.90]
interpretations=["Near identical","Very similar","Related","Somewhat related","Not related"]
for dist,interp in zip(distances,interpretations):
  similarity=1-dist
  print(f"{dist:<15.2f}{similarity:<15.2f}{interp:<20}")

DISTANCE TO SIMILARITY CONVERSION
Distance       Similarity     Interpretation      
0.05           0.95           Near identical      
0.20           0.80           Very similar        
0.40           0.60           Related             
0.65           0.35           Somewhat related    
0.90           0.10           Not related         


In [ ]:
notes_df = pd.read_csv("/content/drive/MyDrive/Copy of college_notes.csv")
print("dataset loaded")
print(f"Shape:{notes_df.shape}")
print(f"Columns:{notes_df.columns.tolist()}")
notes_df.head()
#

dataset loaded
Shape:(15, 4)
Columns:['note_id', 'subject', 'topic', 'content']


,note_id,subject,topic,content
0,N001,Data Engineering,ETL Pipelines,ETL stands for Extract Transform Load. It is t...
1,N002,Data Engineering,SQL Databases,A database is an organized collection of data ...
2,N003,Data Engineering,Data Cleaning,Data cleaning involves fixing or removing inco...
3,N004,Data Engineering,APIs and Data Collection,An API or Application Programming Interface al...
4,N005,Data Engineering,Big Data and PySpark,Big Data refers to extremely large datasets th...


In [ ]:
print("Notes per subject: ")
print(notes_df['subject'].value_counts())


Notes per subject: 
subject
Data Engineering      5
Machine Learning      5
Generative AI         3
Python Programming    2
Name: count, dtype: int64


In [ ]:
#iloc[0] selects the first row by posiiton (index 0)

first_note = notes_df.iloc[0]

print(f"Note ID :{first_note['note_id']}")
print(f"Subject :{first_note['subject']}")
print(f"Topic :{first_note['topic']}")
print(f"Content :{first_note['content']}")
#

Note ID :N001
Subject :Data Engineering
Topic :ETL Pipelines
Content :ETL stands for Extract Transform Load. It is the process of collecting raw data from different sources transforming it into a clean and structured format and loading it into a database or data warehouse for analysis.


In [ ]:
#Extract the data we need for ChromaDB

all_documents =notes_df['content'].tolist()
all_ids = notes_df['note_id'].tolist()
all_metadatas = [
    {"Subject": row['subject'], "Topic": row['topic']}
    for i, row in notes_df.iterrows()
]

print(f"DOcuments prepared :{len(all_documents)}")
print(f"IDs prepared :{len(all_ids)}")
print(f"Metadatas prepared :{len(all_metadatas)}")
print()
print("Smaple ID:",all_ids[0])
print("Sample Metadata:",all_metadatas[0])
print("Sample document (first 80 chars):",all_documents[0][:80])
#

DOcuments prepared :15
IDs prepared :15
Metadatas prepared :15

Smaple ID: N001
Sample Metadata: {'Subject': 'Data Engineering', 'Topic': 'ETL Pipelines'}
Sample document (first 80 chars): ETL stands for Extract Transform Load. It is the process of collecting raw data 
